In [1]:
import os
import json
import uuid
import datetime
from typing import Optional

# Install necessary libraries
!pip install -qU langchain-groq langchain-community langgraph tiktoken

# ---------------------------------------------------------------------------
# LangChain / AI imports
# ---------------------------------------------------------------------------
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ============================= CONFIGURATION ================================

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your_groq_api_key_here")

ISSUE_CATEGORIES = [
    "Billing & Payments",
    "Technical Support",
    "Account Management",
    "Product Information",
    "Shipping & Delivery",
    "Returns & Refunds",
    "Security & Privacy",
    "General Inquiry",
]

PRIORITY_LEVELS = ["Low", "Medium", "High", "Critical"]

# ======================== KNOWLEDGE BASE DOCUMENTS ==========================

SUPPORT_KNOWLEDGE_BASE = [
    # --- Billing & Payments ---
    {
        "title": "Payment Methods",
        "content": (
            "We accept Visa, MasterCard, American Express, PayPal, and bank "
            "transfers. Payments are processed within 1-2 business days. For "
            "corporate accounts, we also offer NET-30 invoicing. All transactions "
            "are encrypted with TLS 1.3 and PCI-DSS compliant."
        ),
        "category": "Billing & Payments",
    },
    {
        "title": "Refund Policy",
        "content": (
            "Refunds are processed within 5-7 business days after approval. "
            "Digital products can be refunded within 14 days of purchase if "
            "unused. Physical products must be returned in original packaging "
            "within 30 days. Shipping costs for returns are covered by the "
            "customer unless the item is defective."
        ),
        "category": "Billing & Payments",
    },
    {
        "title": "Subscription Billing",
        "content": (
            "Subscriptions are billed on the first of each month. You can "
            "upgrade or downgrade your plan at any time from Account Settings. "
            "If you downgrade mid-cycle, the change takes effect at the next "
            "billing period. Cancellations must be made 48 hours before the "
            "renewal date to avoid charges."
        ),
        "category": "Billing & Payments",
    },
    # --- Technical Support ---
    {
        "title": "Password Reset",
        "content": (
            "To reset your password: 1) Go to the login page and click "
            "'Forgot Password'. 2) Enter your registered email address. "
            "3) Check your inbox for a reset link (valid for 1 hour). "
            "4) Click the link and set a new password (minimum 8 characters, "
            "must include a number and special character). If you don't "
            "receive the email, check your spam folder or contact support."
        ),
        "category": "Technical Support",
    },
    {
        "title": "API Rate Limits",
        "content": (
            "Our API enforces the following rate limits: Free tier — 100 "
            "requests/minute, Pro tier — 1,000 requests/minute, Enterprise — "
            "10,000 requests/minute. Exceeding limits returns HTTP 429. "
            "Implement exponential backoff for retries. Batch endpoints are "
            "available for bulk operations to reduce call count."
        ),
        "category": "Technical Support",
    },
    {
        "title": "System Requirements",
        "content": (
            "Minimum requirements: Windows 10/macOS 12/Ubuntu 20.04, 8 GB RAM, "
            "2 GHz dual-core processor, 500 MB free disk space. Recommended: "
            "16 GB RAM, SSD storage. Supported browsers: Chrome 90+, Firefox "
            "88+, Safari 14+, Edge 90+. Mobile apps require iOS 15+ or "
            "Android 11+."
        ),
        "category": "Technical Support",
    },
    {
        "title": "Integration Setup",
        "content": (
            "To integrate with third-party services: 1) Navigate to Settings > "
            "Integrations. 2) Select the service (Slack, Jira, Salesforce, etc.). "
            "3) Authorize via OAuth 2.0. 4) Configure webhook URLs and event "
            "triggers. Detailed API documentation is at docs.example.com/api. "
            "For SSO setup, we support SAML 2.0 and OpenID Connect."
        ),
        "category": "Technical Support",
    },
    # --- Account Management ---
    {
        "title": "Account Deletion",
        "content": (
            "To delete your account: go to Settings > Account > Delete Account. "
            "This action is irreversible. All data will be permanently removed "
            "within 30 days per our data retention policy. Active subscriptions "
            "must be cancelled first. You will receive a confirmation email. "
            "Enterprise accounts must contact their admin for deletion."
        ),
        "category": "Account Management",
    },
    {
        "title": "Two-Factor Authentication",
        "content": (
            "Enable 2FA in Settings > Security > Two-Factor Authentication. "
            "We support authenticator apps (Google Authenticator, Authy) and "
            "SMS verification. Backup codes are generated during setup — store "
            "them securely. If you lose access to your 2FA device, contact "
            "support with your account recovery key."
        ),
        "category": "Account Management",
    },
    # --- Product Information ---
    {
        "title": "Product Plans & Pricing",
        "content": (
            "We offer three plans: Starter ($9/month) — 5 users, 10 GB storage, "
            "email support. Professional ($29/month) — 25 users, 100 GB storage, "
            "priority support, API access. Enterprise (custom pricing) — unlimited "
            "users, unlimited storage, dedicated account manager, SLA guarantee, "
            "custom integrations. All plans include a 14-day free trial."
        ),
        "category": "Product Information",
    },
    {
        "title": "Feature Comparison",
        "content": (
            "Starter includes: basic dashboard, reports, email notifications. "
            "Professional adds: advanced analytics, custom workflows, team "
            "collaboration, Slack integration, API access. Enterprise adds: "
            "SSO/SAML, audit logs, custom SLA, dedicated infrastructure, "
            "white-labeling, and 24/7 phone support."
        ),
        "category": "Product Information",
    },
    # --- Shipping & Delivery ---
    {
        "title": "Shipping Information",
        "content": (
            "Standard shipping: 5-7 business days ($4.99). Express shipping: "
            "2-3 business days ($12.99). Overnight: next business day ($24.99). "
            "Free shipping on orders over $50. International shipping available "
            "to 40+ countries (7-14 business days). Tracking numbers are emailed "
            "within 24 hours of shipment."
        ),
        "category": "Shipping & Delivery",
    },
    # --- Security & Privacy ---
    {
        "title": "Data Privacy & Compliance",
        "content": (
            "We are GDPR, CCPA, and SOC 2 Type II compliant. Data is encrypted "
            "at rest (AES-256) and in transit (TLS 1.3). We conduct annual "
            "penetration testing and quarterly security audits. Users can "
            "export their data at any time via Settings > Privacy > Data Export. "
            "We do not sell personal data to third parties."
        ),
        "category": "Security & Privacy",
    },
]

# ========================== TICKET STORAGE ==================================


class TicketStore:
    """In-memory ticket storage with persistence to JSON file."""

    def __init__(self, storage_path: str = "tickets.json"):
        self.storage_path = storage_path
        self.tickets: list[dict] = []
        self._load()

    def _load(self):
        if os.path.exists(self.storage_path):
            with open(self.storage_path, "r", encoding="utf-8") as f:
                self.tickets = json.load(f)

    def _save(self):
        with open(self.storage_path, "w", encoding="utf-8") as f:
            json.dump(self.tickets, f, indent=2, default=str)

    def create_ticket(
        self,
        summary: str,
        category: str,
        priority: str,
        description: str,
        suggested_solution: str,
        customer_id: Optional[str] = None,
    ) -> dict:
        ticket = {
            "ticket_id": f"TKT-{uuid.uuid4().hex[:8].upper()}",
            "created_at": datetime.datetime.now().isoformat(),
            "status": "Open",
            "customer_id": customer_id or "anonymous",
            "summary": summary,
            "category": category,
            "priority": priority,
            "description": description,
            "suggested_solution": suggested_solution,
        }
        self.tickets.append(ticket)
        self._save()
        return ticket

    def get_ticket(self, ticket_id: str) -> Optional[dict]:
        for t in self.tickets:
            if t["ticket_id"] == ticket_id:
                return t
        return None

    def list_tickets(self, limit: int = 10) -> list[dict]:
        return self.tickets[-limit:]


# =================== DOCUMENT INGESTION & VECTOR STORE ======================


def build_vector_store(documents: list[dict], embeddings) -> FAISS:
    """
    Ingest support knowledge base documents, apply chunking, and build a
    FAISS vector store for retrieval.
    """
    # Convert raw dicts to LangChain Documents
    docs = []
    for doc in documents:
        content = f"Title: {doc['title']}\n\n{doc['content']}"
        metadata = {"title": doc["title"], "category": doc["category"]}
        docs.append(Document(page_content=content, metadata=metadata))

    # Chunking strategy: use RecursiveCharacterTextSplitter for robust splitting
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", ", ", " ", ""],
    )
    split_docs = text_splitter.split_documents(docs)

    # Build FAISS vector store
    vector_store = FAISS.from_documents(split_docs, embeddings)
    return vector_store


# ==================== AGENT TOOLS DEFINITION ================================

# These are module-level references that will be set during initialization.
_vector_store: Optional[FAISS] = None
_ticket_store: Optional[TicketStore] = None
_llm: Optional[ChatGroq] = None


@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the customer support knowledge base for relevant information.
    Use this tool when a customer asks a question that might be answered
    by our documentation — e.g., about billing, account management,
    technical issues, products, shipping, or security.

    Args:
        query: The customer's question or search query.
    """
    if _vector_store is None:
        return "Knowledge base is not initialized."

    try:
        results = _vector_store.similarity_search_with_score(query, k=3)
        if not results:
            return "No relevant information found in the knowledge base."

        formatted = []
        for doc, score in results:
            relevance = "High" if score < 0.5 else "Medium" if score < 0.8 else "Low"
            formatted.append(
                f"[Relevance: {relevance}] "
                f"(Category: {doc.metadata.get('category', 'N/A')})\n"
                f"{doc.page_content}"
            )
        return "\n\n---\n\n".join(formatted)
    except Exception as e:
        return f"Error searching knowledge base: {e}"


@tool
def classify_issue(issue_description: str) -> str:
    """
    Classify a customer issue into a predefined category and assign a priority.
    Use this tool when you need to categorize the customer's problem before
    creating a ticket or providing a structured response.

    Args:
        issue_description: A description of the customer's issue.
    """
    if _llm is None:
        return "LLM is not initialized."

    classification_prompt = f"""Classify the following customer issue into exactly one category
and assign a priority level.

Categories: {', '.join(ISSUE_CATEGORIES)}
Priority Levels: {', '.join(PRIORITY_LEVELS)}

Customer Issue: {issue_description}

Respond in this exact JSON format only (no markdown, no extra text):
{{"category": "<category>", "priority": "<priority>", "reasoning": "<one sentence explanation>"}}
"""
    try:
        response = _llm.invoke(classification_prompt)
        raw_content = response.content

        # Defensive check: ensure content is a non-empty string before trying to parse JSON
        if not isinstance(raw_content, str) or not raw_content.strip():
            # If the LLM returns something that isn't a parseable string,
            # treat it as a failure to classify and directly return the fallback.
            return json.dumps(
                {
                    "category": "General Inquiry",
                    "priority": "Medium",
                    "reasoning": "LLM failed to return valid JSON; defaulting.",
                }
            )

        # Parse the JSON response
        result = json.loads(raw_content.strip())
        # Validate category and priority
        if result["category"] not in ISSUE_CATEGORIES:
            result["category"] = "General Inquiry"
        if result["priority"] not in PRIORITY_LEVELS:
            result["priority"] = "Medium"
        return json.dumps(result, indent=2)
    except (json.JSONDecodeError, KeyError) as e:
        # This block should now catch actual JSON parsing errors or missing keys
        return json.dumps(
            {
                "category": "General Inquiry",
                "priority": "Medium",
                "reasoning": "Could not confidently classify — defaulting due to JSON error or missing key.",
            }
        )
    except Exception as e:
        # Catch any other unexpected errors
        return f"Error classifying issue: {e}"


@tool
def create_support_ticket(
    summary: str,
    category: str,
    priority: str,
    description: str,
    suggested_solution: str,
) -> str:
    """
    Create a support ticket for a customer issue. Use this tool once you have
    classified the issue and gathered enough context from the conversation to
    file a proper ticket.

    Args:
        summary: A concise one-line summary of the issue.
        category: The issue category (e.g., 'Technical Support', 'Billing & Payments').
        priority: The priority level (Low, Medium, High, Critical).
        description: A detailed description of the issue including relevant context.
        suggested_solution: A recommended solution or next step based on the knowledge base.
    """
    if _ticket_store is None:
        return "Ticket store is not initialized."

    try:
        # Validate inputs
        if category not in ISSUE_CATEGORIES:
            category = "General Inquiry"
        if priority not in PRIORITY_LEVELS:
            priority = "Medium"

        ticket = _ticket_store.create_ticket(
            summary=summary,
            category=category,
            priority=priority,
            description=description,
            suggested_solution=suggested_solution,
        )

        return (
            f"Ticket created successfully!\n\n"
            f"  Ticket ID   : {ticket['ticket_id']}\n"
            f"  Status      : {ticket['status']}\n"
            f"  Category    : {ticket['category']}\n"
            f"  Priority    : {ticket['priority']}\n"
            f"  Summary     : {ticket['summary']}\n"
            f"  Created At  : {ticket['created_at']}\n"
            f"  Solution    : {ticket['suggested_solution']}\n"
        )
    except Exception as e:
        return f"Error creating ticket: {e}"


@tool
def lookup_ticket(ticket_id: str) -> str:
    """
    Look up an existing support ticket by its ID.
    Use this when a customer asks about the status of a previously created ticket.

    Args:
        ticket_id: The ticket ID (e.g., TKT-A1B2C3D4).
    """
    if _ticket_store is None:
        return "Ticket store is not initialized."

    ticket = _ticket_store.get_ticket(ticket_id.strip().upper())
    if ticket is None:
        return f"No ticket found with ID '{ticket_id}'. Please verify the ID."
    return json.dumps(ticket, indent=2, default=str)


# ====================== AGENT CONSTRUCTION ==================================


SYSTEM_PROMPT = """You are an intelligent customer support agent for a software company.
Your responsibilities:
1. Answer customer questions using the knowledge base.
2. Classify issues into the correct category and priority.
3. Create support tickets when needed.
4. Maintain a professional, helpful, and empathetic tone.

Workflow for handling customer issues:
- First, search the knowledge base to find relevant information.
- If the customer reports a problem, classify the issue.
- If the issue requires follow-up, create a support ticket.
- Always provide a structured response with the issue summary, category, and suggested solution.

Guidelines:
- If you cannot find an answer in the knowledge base, be honest and offer to escalate.
- Never fabricate information — only use what the knowledge base provides.
- For ambiguous queries, ask clarifying questions before classifying or creating tickets.
- Always confirm with the customer before creating a ticket.

Response format for resolved queries:
  Issue Summary: <brief summary>
  Category: <category>
  Suggested Solution: <solution based on KB>

Response format when creating a ticket:
  Ticket ID: <id>
  Issue Summary: <summary>
  Category: <category>
  Priority: <priority>
  Status: Open
  Suggested Next Steps: <steps>
"""


class ConversationMemory:
    """Simple sliding-window conversation memory."""

    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages: list = []

    def add_user_message(self, content: str):
        self.messages.append(HumanMessage(content=content))
        self._trim()

    def add_ai_message(self, content: str):
        self.messages.append(AIMessage(content=content))
        self._trim()

    def get_messages(self) -> list:
        return list(self.messages)

    def _trim(self):
        # Keep only the last max_turns * 2 messages (user + ai pairs)
        max_messages = self.max_turns * 2
        if len(self.messages) > max_messages:
            self.messages = self.messages[-max_messages:]


def build_agent(llm: ChatGroq, tools: list):
    """Build a ReAct agent with tools using langgraph."""
    agent = create_react_agent(
        model=llm,
        tools=tools,
        prompt=SYSTEM_PROMPT,
    )
    return agent


# ============================= MAIN =========================================


def initialize_system():
    """Initialize all components: LLM, embeddings, vector store, ticket store, agent."""
    global _vector_store, _ticket_store, _llm

    print("=" * 60)
    print("  Intelligent Customer Support & Ticketing System")
    print("=" * 60)
    print("\nInitializing components...")

    # --- LLM ---
    _llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_API_KEY,
    )
    print("  [OK] LLM initialized (llama-3.3-70b-versatile via Groq)")

    # --- Embeddings & Vector Store ---
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    _vector_store = build_vector_store(SUPPORT_KNOWLEDGE_BASE, embeddings)
    print(f"  [OK] Knowledge base loaded ({len(SUPPORT_KNOWLEDGE_BASE)} articles)")

    # --- Ticket Store ---
    _ticket_store = TicketStore()
    print(f"  [OK] Ticket store ready ({len(_ticket_store.tickets)} existing tickets)")

    # --- Agent ---
    tools = [
        search_knowledge_base,
        classify_issue,
        create_support_ticket,
        lookup_ticket,
    ]
    agent = build_agent(_llm, tools)
    print("  [OK] Agent ready with tools:", [t.name for t in tools])
    print()

    return agent


def run_chat_loop(agent):
    """Interactive chat loop with the support agent."""
    print("Welcome! I'm your AI Customer Support Assistant.")
    print("Type your question or describe your issue. Type 'quit' to exit.\n")

    memory = ConversationMemory(max_turns=10)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "q"):
            print("\nThank you for contacting support. Goodbye!")
            break

        try:
            # Build message list with conversation history
            memory.add_user_message(user_input)
            messages = memory.get_messages()

            result = agent.invoke({"messages": messages})
            # Extract the last AI message from the result
            ai_messages = [
                m for m in result["messages"]
                if isinstance(m, AIMessage) and m.content
            ]
            if ai_messages:
                response = ai_messages[-1].content
            else:
                response = "I'm sorry, I couldn't process that."

            memory.add_ai_message(response)
            print(f"\nAgent: {response}\n")
        except Exception as e:
            # Fallback for any unhandled error
            print(
                f"\nAgent: I apologize, but I encountered an unexpected error. "
                f"Please try rephrasing your question or type 'quit' to exit.\n"
                f"(Debug: {e})\n"
            )


def main():
    """Entry point."""
    if GROQ_API_KEY == "your-api-key-here":
        print(
            "ERROR: Please set the GROQ_API_KEY environment variable or "
            "update it in the script before running."
        )
        return

    agent = initialize_system()
    run_chat_loop(agent)


if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
  Intelligent Customer Support & Ticketing System

Initializing components...
  [OK] LLM initialized (llama-3.3-70b-versatile via Groq)


/tmp/ipykernel_5676/3342691459.py:540: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

In [3]:
# ========================== INSTALL DEPENDENCIES ==========================
!pip install -qU langchain-groq langchain-community langchain-huggingface langgraph tiktoken faiss-cpu

# ========================== IMPORTS ==========================
import os
import json
import uuid
import datetime
from typing import Optional

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ========================== CONFIG ==========================
GROQ_API_KEY = "your_groq_api_key_here"

ISSUE_CATEGORIES = [
    "Billing & Payments", "Technical Support", "Account Management",
    "Product Information", "Shipping & Delivery",
    "Returns & Refunds", "Security & Privacy", "General Inquiry"
]

PRIORITY_LEVELS = ["Low", "Medium", "High", "Critical"]

# ========================== KNOWLEDGE BASE ==========================
SUPPORT_KNOWLEDGE_BASE = [
    {"title": "Password Reset",
     "content": "Go to login → Forgot Password → email → reset link (1 hour valid).",
     "category": "Technical Support"},

    {"title": "Refund Policy",
     "content": "Refunds processed in 5–7 days. Digital: 14 days. Physical: 30 days.",
     "category": "Billing & Payments"},

    {"title": "Shipping",
     "content": "Standard 5–7 days, Express 2–3 days, Overnight available.",
     "category": "Shipping & Delivery"},
]

# ========================== TICKET STORE ==========================
class TicketStore:
    def __init__(self):
        self.tickets = []

    def create_ticket(self, summary, category, priority, description, solution):
        ticket = {
            "ticket_id": f"TKT-{uuid.uuid4().hex[:6]}",
            "status": "Open",
            "summary": summary,
            "category": category,
            "priority": priority,
            "description": description,
            "solution": solution,
            "created_at": str(datetime.datetime.now())
        }
        self.tickets.append(ticket)
        return ticket

# ========================== VECTOR STORE ==========================
def build_vector_store():
    docs = []
    for d in SUPPORT_KNOWLEDGE_BASE:
        docs.append(Document(
            page_content=d["title"] + "\n" + d["content"],
            metadata={"category": d["category"]}
        ))

    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    split_docs = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    return FAISS.from_documents(split_docs, embeddings)

# ========================== GLOBALS ==========================
_vector_store = None
_ticket_store = None
_llm = None

# ========================== TOOLS ==========================
@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for relevant support information."""
    if _vector_store is None:
        return "Knowledge base not initialized."

    results = _vector_store.similarity_search(query, k=2)
    if not results:
        return "No results found."

    return "\n\n".join([doc.page_content for doc in results])


@tool
def classify_issue(issue: str) -> str:
    """Classify a customer issue into category and priority."""
    if _llm is None:
        return "LLM not initialized."

    prompt = f"""
Classify issue into category + priority.

Categories: {ISSUE_CATEGORIES}
Priorities: {PRIORITY_LEVELS}

Issue: {issue}

Return JSON:
{{"category":"", "priority":""}}
"""
    try:
        return _llm.invoke(prompt).content
    except Exception as e:
        return f"Classification error: {e}"


@tool
def create_ticket(summary: str, category: str, priority: str, description: str, solution: str) -> str:
    """Create a support ticket for a customer issue."""
    if _ticket_store is None:
        return "Ticket system not initialized."

    ticket = _ticket_store.create_ticket(summary, category, priority, description, solution)
    return f"✅ Ticket Created: {ticket['ticket_id']}"

# ========================== SYSTEM PROMPT ==========================
SYSTEM_PROMPT = """
You are a customer support AI.

Steps:
1. Search knowledge base
2. Classify issue
3. Suggest solution
4. Ask before creating ticket

Be helpful and professional.
"""

# ========================== AGENT ==========================
def initialize():
    global _vector_store, _ticket_store, _llm

    print("Initializing system...")

    _llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_API_KEY
    )

    _vector_store = build_vector_store()
    _ticket_store = TicketStore()

    tools = [search_knowledge_base, classify_issue, create_ticket]

    agent = create_react_agent(
        model=_llm,
        tools=tools,
        prompt=SYSTEM_PROMPT
    )

    return agent

# ========================== CHAT LOOP ==========================
def chat():
    agent = initialize()
    memory = []

    print("\n🤖 AI Support Ready (type 'quit' to exit)\n")

    while True:
        try:
            user = input("You: ")

            if user.lower() == "quit":
                print("Goodbye 👋")
                break

            memory.append(HumanMessage(content=user))

            result = agent.invoke({"messages": memory})

            # Safe extraction
            if "messages" in result and len(result["messages"]) > 0:
                response = result["messages"][-1].content
            else:
                response = "Sorry, I couldn't process that."

            print("Agent:", response)

            memory.append(AIMessage(content=response))

        except Exception as e:
            print(f"⚠ Error: {e}")

# ========================== RUN ==========================
chat()

Initializing system...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_5676/3294837395.py:162: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



🤖 AI Support Ready (type 'quit' to exit)

You: what payment method do you accept 
⚠ Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search_knowledge_base{"query": "accepted payment methods"}</function>'}}
You: quite
Agent: We accept various payment methods, including credit cards (Visa, Mastercard, Amex), PayPal, and bank transfers. Would you like to know more about our payment options or is there anything else I can assist you with?


KeyboardInterrupt: Interrupted by user

In [5]:
# ========================== INSTALL ==========================
!pip install -qU langchain-groq langchain-community langchain-huggingface faiss-cpu

# ========================== IMPORTS ==========================
import uuid
import datetime

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# ========================== CONFIG ==========================
GROQ_API_KEY = "your_groq_api_key_here"

# ========================== KNOWLEDGE BASE ==========================
SUPPORT_KNOWLEDGE_BASE = [
    {"title": "Payment Methods",
     "content": "We accept Visa, MasterCard, PayPal, and bank transfers."},

    {"title": "Password Reset",
     "content": "Go to login → Forgot Password → email → reset link."},

    {"title": "Refund Policy",
     "content": "Refunds processed in 5–7 days. Digital: 14 days."},
]

# ========================== VECTOR STORE ==========================
def build_vector_store():
    docs = [Document(page_content=d["title"] + "\n" + d["content"]) for d in SUPPORT_KNOWLEDGE_BASE]

    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    docs = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    return FAISS.from_documents(docs, embeddings)

# ========================== TICKET SYSTEM ==========================
tickets = []

def create_ticket(issue, solution):
    ticket = {
        "id": f"TKT-{uuid.uuid4().hex[:6]}",
        "issue": issue,
        "solution": solution,
        "status": "Open",
        "time": str(datetime.datetime.now())
    }
    tickets.append(ticket)
    return ticket

# ========================== INIT ==========================
print("Initializing system...")

vector_store = build_vector_store()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

# ========================== CHAT ==========================
print("\n🤖 AI Support Ready\n")

while True:
    user = input("You: ")

    if user.lower() == "quit":
        break

    # 🔍 RETRIEVE (RAG)
    docs = vector_store.similarity_search(user, k=2)
    context = "\n".join([d.page_content for d in docs])

    # 🧠 GENERATE
    prompt = f"""
You are a customer support assistant.

Use the context to answer.

Context:
{context}

User:
{user}

Answer clearly and professionally.
"""

    response = llm.invoke(prompt).content

    print("Agent:", response)

    # 🎫 OPTIONAL ticket trigger
    if "problem" in user.lower() or "issue" in user.lower():
        ticket = create_ticket(user, response)
        print(f"✅ Ticket Created: {ticket['id']}")

Initializing system...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🤖 AI Support Ready

You: what payment method do you accept
Agent: We accept the following payment methods: 

1. Visa
2. MasterCard
3. PayPal
4. Bank transfers

If you have any further questions or need assistance with a payment, please don't hesitate to ask.
You: How long does a refund take?
Agent: Our refund processing time typically takes 5-7 business days for most refunds. However, if your refund is for a digital product, please allow 14 days for the refund to be processed. If you have any further questions or concerns, feel free to ask.
You: How can I pay for your service?
Agent: We offer a variety of payment methods for your convenience. You can pay for our service using Visa, MasterCard, PayPal, or by bank transfer. If you have any specific questions or need further assistance with the payment process, please don't hesitate to ask.
You: What happens if I request a refund after 20 days?
Agent: According to our Refund Policy, the refund timeframe varies depending on the type of pr

In [6]:
!pip install -qU langchain langchain-community langchain-huggingface faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.6 MB/s eta 0:00:00


In [7]:
# ========================== IMPORTS ==========================
import os
import json
import uuid
import numpy as np

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_groq import ChatGroq

# ========================== CONFIG ==========================
GROQ_API_KEY = "PUT_YOUR_KEY_HERE"

# ========================== FUNCTIONS ==========================

# 📄 Load Resume
def load_resume(file_path):
    try:
        loader = PyPDFLoader(file_path)
        pages = loader.load()
        return " ".join([p.page_content for p in pages])
    except:
        return ""

# 🧾 Create Vector Store
def create_vector_store(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    docs = splitter.create_documents([text])
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    return FAISS.from_documents(docs, embeddings)

# 🔍 Semantic Similarity
def compute_similarity(store, job_desc):
    results = store.similarity_search_with_score(job_desc, k=3)
    scores = [1 - score for _, score in results]
    return round(float(np.mean(scores)) * 100, 2)

# 🧠 LLM Evaluation
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

def evaluate_candidate(resume_text, job_desc):
    prompt = f"""
You are an HR expert.

Compare this resume with job requirements.

Job Description:
{job_desc}

Resume:
{resume_text}

Return ONLY JSON:
{{
"match_score": number (0-100),
"skills_matched": [],
"missing_skills": [],
"experience_level": "",
"recommendation": "Hire / Consider / Reject"
}}
"""
    try:
        return llm.invoke(prompt).content
    except Exception as e:
        return f"Error: {e}"

# 🏆 Ranking System
def rank_candidates(resumes, job_desc):
    results = []

    for name, text in resumes.items():

        if not text.strip():
            print(f"⚠ Skipping {name}: empty resume")
            continue

        store = create_vector_store(text)
        sim_score = compute_similarity(store, job_desc)
        evaluation = evaluate_candidate(text, job_desc)

        results.append({
            "name": name,
            "similarity_score": sim_score,
            "evaluation": evaluation
        })

    return sorted(results, key=lambda x: x["similarity_score"], reverse=True)

# ========================== INPUT ==========================

# 🧾 Job Description
job_description = """
We are hiring a Machine Learning Engineer.

Requirements:
- Python
- Machine Learning
- Deep Learning
- NLP
- PyTorch or TensorFlow
- 2+ years experience
"""

# 📄 Upload resumes in Colab before running
resumes = {
    "Candidate_A": load_resume("/content/resume1.pdf"),
    "Candidate_B": load_resume("/content/resume2.pdf"),
}

# ========================== RUN ==========================
print("🚀 Running AI Hiring System...\n")

ranked = rank_candidates(resumes, job_description)

# ========================== OUTPUT ==========================
for r in ranked:
    print("\n==============================")
    print("👤 Name:", r["name"])
    print("📊 Similarity Score:", r["similarity_score"])
    print("🧠 Evaluation:")
    print(r["evaluation"])

🚀 Running AI Hiring System...

⚠ Skipping Candidate_A: empty resume
⚠ Skipping Candidate_B: empty resume
